# Cookie Cutter — notebook twin of the anyui UI

Same tree as `cutter_anyui.html` / `cutter_anyui_main.js`:

```
VBox
  HBox  toolbar (Shape, Scale, Fit, Insert, Delete, Export, Spin)
  HBox  stage
    CurveEditorWidget     → es6/curve-editor-widget.js
    WebGLCutterWidget     → es6/webgl-cutter-widget.js
  PathTableWidget         → es6/path-table.js
```

Chrome is **ipywidgets** (`Box` / `HBox` / `VBox` / `Button` / `Dropdown` / `FloatText`).
The two canvases and the segment table are **anywidget** classes pointed at the existing ES6 render modules.

`turtlePath` is `[[length, angleDegrees], …]` plus `startPoint`, `startAngle`, `name`.

Requires `pip install anywidget ipywidgets` in the kernel. Open this notebook from `cutter-es6/` (or any cwd that can see that folder). `standalone.html` is unchanged.


In [1]:
from __future__ import annotations

import contextlib
import io
import json
import sys
from copy import deepcopy
from pathlib import Path

from IPython.display import display

try:
    import anywidget
    import ipywidgets as W
    import traitlets
except ImportError as exc:
    raise ImportError(
        "This notebook needs anywidget + ipywidgets in the kernel.\n"
        "    pip install anywidget ipywidgets"
    ) from exc


def find_cutter_root() -> Path:
    here = Path.cwd().resolve()
    extra = []
    if "__file__" in globals():
        extra.append(Path(globals()["__file__"]).resolve().parent)
    seen: list[Path] = []
    for start in [here, *extra]:
        for p in [start, *start.parents]:
            if p not in seen:
                seen.append(p)
    for p in seen:
        for cand in (p, p / "cutter-es6"):
            if (cand / "es6" / "curve-editor-widget.js").is_file():
                return cand
    raise FileNotFoundError(
        "Cannot find cutter-es6/es6/curve-editor-widget.js. "
        "cd into cutter-es6 (or a parent) and re-run."
    )


ROOT = find_cutter_root()
ES6 = ROOT / "es6"
sys.path.insert(0, str(ROOT))

from es6_to_iife_anyui import convertES6toIIFE  # project's own bundler


def bundle_esm(entry: Path) -> str:
    """Turn an ES6 widget file + relative imports into one anywidget ESM string."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        iife, _css = convertES6toIIFE(
            content=entry.read_text(encoding="utf-8"),
            module_dir=str(entry.parent),
            module_filename=entry.name,
            minify=False,
        )
    key = json.dumps(entry.name)
    # convertES6toIIFE attaches exports on window.modules[filename].
    return (
        iife
        + "\nconst __cutterMod = (globalThis.modules && globalThis.modules["
        + key
        + "]) || {};\n"
        + "export const render = __cutterMod.render "
        + "|| (__cutterMod.default && __cutterMod.default.render);\n"
        + "export default { render };\n"
    )


ESM_EDITOR = bundle_esm(ES6 / "curve-editor-widget.js")
ESM_VIEWER = bundle_esm(ES6 / "webgl-cutter-widget.js")
ESM_TABLE = bundle_esm(ES6 / "path-table.js")

WIDGET_CSS = """
.cutter-anyui-host { min-height: 260px; height: 320px; width: 100%; position: relative; }
.cutter-anyui-host canvas { width: 100%; height: 100%; display: block; touch-action: none; background: #12110f; }
.path-table-wrap { max-height: 240px; overflow: auto; width: 100%; padding: 6px 4px 12px; }
.path-stats { margin: 0 0 8px; font-size: 12px; color: #666; }
.path-table-wrap table { width: 100%; border-collapse: collapse; font-variant-numeric: tabular-nums; font-size: 13px; }
.path-table-wrap th { text-align: left; color: #666; font-weight: 500; padding: 4px 6px; }
.path-table-wrap td { padding: 2px 6px; }
.path-table-wrap td input { width: 100%; padding: 4px 6px; }
.path-table-wrap tr.sel td { background: #d7ebe6; }
"""


BLANK = {
    "name": "Custom",
    "startPoint": [0.0, 0.0],
    "startAngle": 0.0,
    "turtlePath": [],
}


def _load_outlines() -> tuple[list[str], dict]:
    names, outlines = [], {}
    try:
        import subprocess

        src = (
            "import { getOutline, OUTLINE_NAMES } from './es6/cookiecutters.js';\n"
            "const outlines = Object.fromEntries(OUTLINE_NAMES.map(n => [n, getOutline(n)]));\n"
            "console.log(JSON.stringify({ names: OUTLINE_NAMES, outlines }));\n"
        )
        raw = subprocess.check_output(
            ["node", "--input-type=module", "-e", src],
            cwd=str(ROOT),
            text=True,
        )
        data = json.loads(raw)
        names = [n for n in data["names"] if n != "Blade"]
        outlines = data["outlines"]
    except Exception:
        # Fallback subset so the notebook still boots without node.
        outlines = {
            "Duck": {
                "name": "Duck",
                "startPoint": [0, 0],
                "startAngle": 180,
                "turtlePath": [
                    [0.4, -10], [13.297, 25], [3, -80], [4, 160],
                    [22.913, 90], [15, 90], [5, -90], [5, 20],
                    [3, 170], [2, -20], [3, -90], [15, 220], [5, -125],
                ],
            },
            "Heart": {
                "name": "Heart",
                "startPoint": [0, 0],
                "startAngle": 180,
                "turtlePath": [
                    [0.45, -45], [10, 180], [6.91, -10], [1.1, 110],
                    [6.91, -10], [10, 180], [0.45, -45],
                ],
            },
            "Star": {
                "name": "Star",
                "startPoint": [0, 0],
                "startAngle": 0,
                "turtlePath": [
                    [2, -58], [8, 0], [3.2, 130], [8, 0],
                ] * 5,
            },
        }
        names = list(outlines.keys())
    names = names + (["Blank"] if "Blank" not in names else [])
    outlines["Blank"] = deepcopy(BLANK)
    return names, outlines


SHAPE_NAMES, OUTLINES = _load_outlines()
print(f"root: {ROOT}")
print(f"shapes: {', '.join(SHAPE_NAMES)}")
print(f"bundled ESM: editor {len(ESM_EDITOR)} B, viewer {len(ESM_VIEWER)} B, table {len(ESM_TABLE)} B")


root: /private/var/mobile/Containers/Data/Application/96B0A1A4-0205-4CE1-BABA-474BC9B9139A/Documents/Grok/cutter-es6
shapes: Duck, Heart, Star, Blank
bundled ESM: editor 28435 B, viewer 30366 B, table 12725 B


In [2]:
class CurveEditorWidget(anywidget.AnyWidget):
    """anywidget twin of es6/curve-editor-cls.js."""

    _esm = ESM_EDITOR
    _css = WIDGET_CSS
    name = traitlets.Unicode("Custom").tag(sync=True)
    startPoint = traitlets.List(traitlets.Float(), default_value=[0.0, 0.0]).tag(sync=True)
    startAngle = traitlets.Float(0.0).tag(sync=True)
    turtlePath = traitlets.List(default_value=[]).tag(sync=True)
    selected_index = traitlets.Int(-1).tag(sync=True)

    def get_outline(self) -> dict:
        return {
            "name": self.name,
            "startPoint": list(self.startPoint),
            "startAngle": float(self.startAngle),
            "turtlePath": [list(seg) for seg in (self.turtlePath or [])],
        }

    def set_outline(self, outline: dict, *, keep_selection: bool = True) -> None:
        path = [list(seg) for seg in (outline.get("turtlePath") or [])]
        self.name = outline.get("name") or "Custom"
        self.startPoint = [float(x) for x in (outline.get("startPoint") or [0, 0])]
        self.startAngle = float(outline.get("startAngle") or 0)
        self.turtlePath = path
        n = len(path)
        if not keep_selection or self.selected_index >= n:
            self.selected_index = n - 1 if n else -1

    def insert_segment(self) -> int:
        # Same rule as CurveEditor.insertSegment: [4, 0] after the highlight.
        path = [list(seg) for seg in (self.turtlePath or [])]
        n = len(path)
        i = self.selected_index
        at = (i + 1) if i >= 0 else n
        at = max(0, min(at, n))
        path.insert(at, [4.0, 0.0])
        self.turtlePath = path
        self.selected_index = at
        # Path mutation is the source of truth; the canvas syncs via change:turtlePath.
        # Do not also send cmd:insert — that would insert twice.
        return at

    def delete_segment(self) -> int:
        path = [list(seg) for seg in (self.turtlePath or [])]
        n = len(path)
        if not n:
            self.selected_index = -1
            return -1
        i = self.selected_index if self.selected_index >= 0 else n - 1
        if i < 0 or i >= n:
            return self.selected_index
        path.pop(i)
        self.turtlePath = path
        self.selected_index = min(i, len(path) - 1) if path else -1
        return self.selected_index

    def fit(self) -> None:
        self.send({"cmd": "fit"})


class WebGLCutterWidget(anywidget.AnyWidget):
    """anywidget twin of es6/webgl-cutter-cls.js."""

    _esm = ESM_VIEWER
    _css = WIDGET_CSS
    name = traitlets.Unicode("Custom").tag(sync=True)
    startPoint = traitlets.List(traitlets.Float(), default_value=[0.0, 0.0]).tag(sync=True)
    startAngle = traitlets.Float(0.0).tag(sync=True)
    turtlePath = traitlets.List(default_value=[]).tag(sync=True)
    outlineScale = traitlets.Float(11.0).tag(sync=True)
    bladeScale = traitlets.Float(5.0).tag(sync=True)
    animate = traitlets.Bool(True).tag(sync=True)

    def set_outline(self, outline: dict, *, scale: float | None = None, blade_scale: float | None = None) -> None:
        self.name = outline.get("name") or "Custom"
        self.startPoint = [float(x) for x in (outline.get("startPoint") or [0, 0])]
        self.startAngle = float(outline.get("startAngle") or 0)
        self.turtlePath = [list(seg) for seg in (outline.get("turtlePath") or [])]
        if scale is not None:
            self.outlineScale = float(scale)
        if blade_scale is not None:
            self.bladeScale = float(blade_scale)


class PathTableWidget(anywidget.AnyWidget):
    """anywidget twin of es6/path-table-cls.js."""

    _esm = ESM_TABLE
    _css = WIDGET_CSS
    name = traitlets.Unicode("Custom").tag(sync=True)
    startPoint = traitlets.List(traitlets.Float(), default_value=[0.0, 0.0]).tag(sync=True)
    startAngle = traitlets.Float(0.0).tag(sync=True)
    turtlePath = traitlets.List(default_value=[]).tag(sync=True)
    selected_index = traitlets.Int(-1).tag(sync=True)


In [3]:
initial = deepcopy(OUTLINES.get("Duck") or next(iter(OUTLINES.values())))
n0 = len(initial["turtlePath"])

editor = CurveEditorWidget(
    name=initial["name"],
    startPoint=list(initial["startPoint"]),
    startAngle=float(initial["startAngle"]),
    turtlePath=[list(s) for s in initial["turtlePath"]],
    selected_index=(n0 - 1) if n0 else -1,
    layout=W.Layout(width="100%", height="320px", min_height="260px", flex="1 1 auto"),
)
viewer = WebGLCutterWidget(
    name=initial["name"],
    startPoint=list(initial["startPoint"]),
    startAngle=float(initial["startAngle"]),
    turtlePath=[list(s) for s in initial["turtlePath"]],
    outlineScale=11.0,
    bladeScale=5.0,
    animate=True,
    layout=W.Layout(width="100%", height="320px", min_height="260px", flex="1 1 auto"),
)
table = PathTableWidget(
    name=initial["name"],
    startPoint=list(initial["startPoint"]),
    startAngle=float(initial["startAngle"]),
    turtlePath=[list(s) for s in initial["turtlePath"]],
    selected_index=editor.selected_index,
    layout=W.Layout(width="100%", max_height="240px"),
)

shape = W.Dropdown(options=SHAPE_NAMES, value="Duck" if "Duck" in SHAPE_NAMES else SHAPE_NAMES[0], description="Shape")
scale = W.FloatText(value=11.0, description="Scale", step=0.5)
btn_fit = W.Button(description="Fit")
btn_insert = W.Button(description="Insert")
btn_delete = W.Button(description="Delete")
btn_export = W.Button(description="Export JSON", button_style="primary")
btn_spin = W.Button(description="Spin")
status = W.HTML(value="<em>Duck loaded — last segment selected. Drag a handle to edit.</em>")

syncing = False


def current_outline() -> dict:
    return editor.get_outline()


def apply_outline(outline: dict, *, keep_selection: bool = True, fit: bool = False) -> None:
    editor.set_outline(outline, keep_selection=keep_selection)
    viewer.set_outline(outline, scale=float(scale.value))
    table.name = editor.name
    table.startPoint = list(editor.startPoint)
    table.startAngle = float(editor.startAngle)
    table.turtlePath = [list(s) for s in editor.turtlePath]
    table.selected_index = editor.selected_index
    btn_delete.disabled = not bool(editor.turtlePath)
    n = len(editor.turtlePath or [])
    status.value = f"<code>{editor.name}</code> · {n} arcs · selected {editor.selected_index}"
    if fit:
        editor.fit()


def with_sync(fn):
    global syncing
    if syncing:
        return
    syncing = True
    try:
        fn()
    finally:
        syncing = False


def push_from_editor(*_):
    def _go():
        outline = current_outline()
        viewer.set_outline(outline, scale=float(scale.value))
        table.name = outline["name"]
        table.startPoint = list(outline["startPoint"])
        table.startAngle = float(outline["startAngle"])
        table.turtlePath = [list(s) for s in outline["turtlePath"]]
        table.selected_index = int(editor.selected_index)
        btn_delete.disabled = not bool(outline["turtlePath"])
        status.value = (
            f"<code>{outline['name']}</code> · {len(outline['turtlePath'])} arcs · "
            f"selected {editor.selected_index}"
        )
    with_sync(_go)


def push_from_table(*_):
    def _go():
        outline = {
            "name": table.name,
            "startPoint": list(table.startPoint),
            "startAngle": float(table.startAngle),
            "turtlePath": [list(s) for s in (table.turtlePath or [])],
        }
        editor.set_outline(outline, keep_selection=True)
        editor.selected_index = int(table.selected_index)
        viewer.set_outline(outline, scale=float(scale.value))
        btn_delete.disabled = not bool(outline["turtlePath"])
    with_sync(_go)


editor.observe(push_from_editor, names=["turtlePath", "startPoint", "startAngle", "name", "selected_index"])
table.observe(push_from_table, names=["turtlePath", "startPoint", "startAngle", "selected_index"])


def on_shape(change):
    name = change["new"]
    outline = deepcopy(OUTLINES.get(name, BLANK))
    if name == "Blank":
        outline = deepcopy(BLANK)
    with_sync(lambda: apply_outline(outline, keep_selection=False, fit=True))


shape.observe(on_shape, names="value")
scale.observe(lambda c: viewer.set_outline(current_outline(), scale=float(c["new"] or 11)), names="value")

btn_fit.on_click(lambda *_: editor.fit())
btn_insert.on_click(lambda *_: editor.insert_segment())
btn_delete.on_click(lambda *_: editor.delete_segment())
btn_spin.on_click(lambda *_: setattr(viewer, "animate", True))


def on_export(_=None):
    text = json.dumps(current_outline(), indent=2)
    print(text)
    status.value = f"<pre style='max-height:12rem;overflow:auto'>{text}</pre>"


btn_export.on_click(on_export)

toolbar = W.HBox(
    [shape, scale, btn_fit, btn_insert, btn_delete, btn_export, btn_spin],
    layout=W.Layout(flex_flow="row wrap", align_items="center"),
)
path_panel = W.VBox(
    [W.HTML("<b>Path</b>"), editor],
    layout=W.Layout(width="50%", min_width="16rem", flex="1 1 16rem"),
)
view_panel = W.VBox(
    [W.HTML("<b>3D · Blade</b>"), viewer],
    layout=W.Layout(width="50%", min_width="16rem", flex="1 1 16rem"),
)
stage = W.HBox([path_panel, view_panel], layout=W.Layout(width="100%", align_items="stretch"))
root = W.VBox(
    [
        W.HTML("<h3 style='margin:0 0 8px'>Cookie Cutter</h3>"),
        toolbar,
        stage,
        status,
        table,
    ],
    layout=W.Layout(width="100%"),
)

btn_delete.disabled = not bool(editor.turtlePath)
display(root)
